In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 1 ▸ Imports & global settings
# ─────────────────────────────────────────────────────────────
from functools import reduce
from pathlib import Path
import warnings

import pandas as pd

# Configuration
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# I/O Paths
EXCEL_PATH = Path("data/raw/DataSheet_ETH_250902.xlsx")
OUTPUT_CSV = Path("data/processed/preprocessed_dataset.csv")  # Standardized name

# Validation
if not EXCEL_PATH.exists():
    raise FileNotFoundError(f"Input Excel file not found at: {EXCEL_PATH}")

# Ensure output directory exists
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print(f"Input:  {EXCEL_PATH}")
print(f"Output: {OUTPUT_CSV}")

Input:  data\raw\DataSheet_ETH_250902.xlsx
Output: data\processed\preprocessed_dataset.csv


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 2 ▸ Helper functions for Excel parsing
# ─────────────────────────────────────────────────────────────

def read_multiheader(sheet_name):
    """
    Read a sheet that uses a two-row header (top: variable block,
    second: ETH-xx run identifiers), returning a DataFrame with a
    MultiIndex for columns.
    """
    try:
        df = pd.read_excel(EXCEL_PATH, sheet_name=sheet_name, header=[0, 1])
    except ValueError as e:
        print(f"Error reading sheet '{sheet_name}': {e}")
        return pd.DataFrame()

    # Normalize column level names (strip whitespace, handle NaNs)
    new_columns = []
    for col in df.columns:
        level0 = str(col[0]).strip()
        level1 = str(col[1]).strip() if not pd.isna(col[1]) and str(col[1]) != 'nan' else ""
        new_columns.append((level0, level1))

    df.columns = pd.MultiIndex.from_tuples(new_columns, names=["var", "run"])
    return df


def get_time_series(df_multi):
    """
    Extract the Time(ms) series from a two-level column DataFrame.
    Assumes 'Time(ms)' is the first column block.
    """
    # Find columns where the top level is 'Time(ms)'
    time_cols = [c for c in df_multi.columns if "Time(ms)" in c[0]]

    if not time_cols:
        raise ValueError("No 'Time(ms)' column found in sheet.")

    # Usually Time is the very first column
    return df_multi[time_cols[0]].astype(float).rename("Time_ms")


def stack_block(df_multi, top_name, value_name):
    """
    For a given top-level column block (e.g., 'Chamber pressure (bar)'),
    produce a long DataFrame with ['run', 'Time_ms', value_name].
    """
    # 1. Get the shared time index
    try:
        time = get_time_series(df_multi)
    except ValueError as e:
        print(f"Skipping block '{top_name}': {e}")
        return pd.DataFrame()

    # 2. Identify columns belonging to this variable block
    #    We look for columns where the top-level header matches `top_name`
    #    and the second-level header (run name) is not empty.
    block_cols = [c for c in df_multi.columns if c[0] == top_name and c[1] != ""]

    if not block_cols:
        print(f"Warning: No columns found for variable '{top_name}'")
        return pd.DataFrame()

    # 3. Extract and reshape
    sub = df_multi[block_cols].copy()
    sub.columns = [c[1] for c in block_cols]  # Drop top level, keep run names

    # Insert time and melt
    sub.insert(0, "Time_ms", time.values)
    long = sub.melt(id_vars="Time_ms", var_name="run", value_name=value_name)

    # 4. Clean types
    long[value_name] = pd.to_numeric(long[value_name], errors="coerce")
    long["Time_ms"] = pd.to_numeric(long["Time_ms"], errors="coerce")

    return long.dropna(subset=["Time_ms"])  # Drop rows where time is invalid


def groupwise_ffill_bfill(df, group_cols, order_cols, cols_to_fill=None):
    """
    Within each group, sort and apply ffill followed by bfill for selected columns.
    """
    df = df.sort_values(group_cols + order_cols).copy()

    if cols_to_fill is None:
        # Default: all numeric columns that are not grouping keys
        keyset = set(group_cols + order_cols)
        cols_to_fill = [
            c for c in df.columns
            if c not in keyset and pd.api.types.is_numeric_dtype(df[c])
        ]

    def _fill(g):
        g[cols_to_fill] = g[cols_to_fill].ffill().bfill()
        return g

    return df.groupby(group_cols, as_index=False, group_keys=False).apply(_fill)

In [13]:
# ─────────────────────────────────────────────────────────────
# Cell 3 ▸ Process Inputs ("Exp. Conditions in Time")
# ─────────────────────────────────────────────────────────────

print("Reading 'Exp. Conditions in Time'...")
ect = read_multiheader("Exp. Conditions in Time")

# Extract each variable block
inp_pc = stack_block(ect, "Chamber pressure (bar)", "Pc_bar")
inp_tc = stack_block(ect, "Chamber temperature (K)", "Tc_K")
inp_pinj = stack_block(ect, "Injection pressure (bar)", "Pinj_bar")
inp_rho = stack_block(ect, "Density (kg/m3)", "rho_kgm3")
inp_mu = stack_block(ect, "Viscosity (Pas)", "mu_Pas")

# List of input dataframes
input_dfs = [inp_pc, inp_tc, inp_pinj, inp_rho, inp_mu]
input_dfs = [d for d in input_dfs if not d.empty]

if not input_dfs:
    raise ValueError("No input data could be extracted!")

# Merge all input blocks on ['run','Time_ms']
# Using outer join to ensure we keep all time points from all variables
inputs = reduce(
    lambda left, right: pd.merge(left, right, on=["run", "Time_ms"], how="outer"),
    input_dfs,
)

# Ensure canonical run names (string)
inputs["run"] = inputs["run"].astype(str)

print(f"Inputs shape: {inputs.shape}")
print(f"Unique runs in inputs: {sorted(inputs['run'].unique())}")
display(inputs.head())

Reading 'Exp. Conditions in Time'...
Inputs shape: (847, 7)
Unique runs in inputs: ['ETH-01', 'ETH-02', 'ETH-03', 'ETH-04', 'ETH-05', 'ETH-06', 'ETH-06.1']


,Time_ms,run,Pc_bar,Tc_K,Pinj_bar,rho_kgm3,mu_Pas
0,0.000,ETH-01,55.03180,192.029519,98.864550,810.720228,0.001879
1,0.025,ETH-01,55.00570,192.015831,98.874062,810.718262,0.001879
2,0.050,ETH-01,55.00810,191.988228,98.907356,810.718443,0.001879
3,0.075,ETH-01,55.01635,192.081988,98.855037,810.719065,0.001879
4,0.100,ETH-01,55.01250,191.988000,98.878819,810.718775,0.001879


In [14]:
# ─────────────────────────────────────────────────────────────
# Cell 4 ▸ Process Targets (Angle & Penetration sheets)
# ─────────────────────────────────────────────────────────────

target_configs = [
    ("Spray Angle (Shadow)", "Smoothed angle (deg)", "angle_shadow_deg"),
    ("Spray Penetration (Shadow)", "Penetration (L/D)", "len_shadow_L_D"),
    ("Spray Angle (Mie)", "Smoothed angle (deg)", "angle_mie_deg"),
    ("Spray Penetration (Mie)", "Penetration (L/D)", "len_mie_L_D"),
]

target_dfs = []

for sheet, col_name, var_name in target_configs:
    print(f"Reading '{sheet}'...")
    df_raw = read_multiheader(sheet)
    if not df_raw.empty:
        df_stack = stack_block(df_raw, col_name, var_name)
        if not df_stack.empty:
            target_dfs.append(df_stack)
        else:
            print(f"  ⚠ Warning: Could not extract '{var_name}' from '{sheet}'")
    else:
        print(f"  ⚠ Warning: Sheet '{sheet}' is empty or missing")

if not target_dfs:
    raise ValueError("No target data could be extracted!")

# Quick check of first target
print(f"\nFirst target dataframe shape: {target_dfs[0].shape}")
display(target_dfs[0].head())

Reading 'Spray Angle (Shadow)'...
Reading 'Spray Penetration (Shadow)'...
Reading 'Spray Angle (Mie)'...Reading 'Spray Angle (Mie)'...
Reading 'Spray Penetration (Mie)'...

First target dataframe shape: (726, 3)

Reading 'Spray Penetration (Mie)'...

First target dataframe shape: (726, 3)


,Time_ms,run,angle_shadow_deg
0,0.000,ETH-01,NaN
1,0.025,ETH-01,NaN
2,0.050,ETH-01,NaN
3,0.075,ETH-01,NaN
4,0.100,ETH-01,NaN


In [15]:
# ─────────────────────────────────────────────────────────────
# Cell 5 ▸ Merge Inputs and Targets
# ─────────────────────────────────────────────────────────────

# Start with inputs
merged = inputs.copy()

# Merge each target dataframe
for tgt_df in target_dfs:
    merged = pd.merge(merged, tgt_df, on=["run", "Time_ms"], how="outer")

# Define column order
cols_order = [
    "run",
    "Time_ms",
    "Pc_bar",
    "Tc_K",
    "Pinj_bar",
    "rho_kgm3",
    "mu_Pas",
    "angle_shadow_deg",
    "len_shadow_L_D",
    "angle_mie_deg",
    "len_mie_L_D",
]

# Filter to keep only columns that exist (in case some targets were missing)
final_cols = [c for c in cols_order if c in merged.columns]
merged = merged[final_cols]

# Coerce all numeric columns
for c in merged.columns:
    if c != "run":
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

print("Merged shape:", merged.shape)
print("Unique runs:", sorted(merged["run"].unique()))
display(merged.head())

Merged shape: (847, 11)
Unique runs: ['ETH-01', 'ETH-02', 'ETH-03', 'ETH-04', 'ETH-05', 'ETH-06', 'ETH-06.1']


,run,Time_ms,Pc_bar,Tc_K,Pinj_bar,rho_kgm3,mu_Pas,angle_shadow_deg,len_shadow_L_D,angle_mie_deg,len_mie_L_D
0,ETH-01,0.000,55.03180,192.029519,98.864550,810.720228,0.001879,NaN,NaN,NaN,NaN
1,ETH-01,0.025,55.00570,192.015831,98.874062,810.718262,0.001879,NaN,NaN,NaN,NaN
2,ETH-01,0.050,55.00810,191.988228,98.907356,810.718443,0.001879,NaN,NaN,NaN,NaN
3,ETH-01,0.075,55.01635,192.081988,98.855037,810.719065,0.001879,NaN,13.126559,NaN,17.571262
4,ETH-01,0.100,55.01250,191.988000,98.878819,810.718775,0.001879,NaN,20.204667,NaN,24.506565


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 6 ▸ Handle Missing Values (Groupwise Interpolation)
# ─────────────────────────────────────────────────────────────

# Identify numeric columns to fill
key_cols = ["run", "Time_ms"]
num_cols = [c for c in merged.columns if c not in key_cols]

print(f"Filling missing values for: {num_cols}")

# Apply groupwise ffill/bfill
filled = groupwise_ffill_bfill(
    merged.copy(),
    group_cols=["run"],
    order_cols=["Time_ms"],
    cols_to_fill=num_cols
)

# Check remaining NaNs
na_counts = filled[num_cols].isna().sum()
if na_counts.sum() > 0:
    print("\n⚠ Warning: Some NaNs remain after interpolation:")
    print(na_counts[na_counts > 0])
else:
    print("\n✔ No missing values remaining.")

filled.head()

Filling missing values for: ['Pc_bar', 'Tc_K', 'Pinj_bar', 'rho_kgm3', 'mu_Pas', 'angle_shadow_deg', 'len_shadow_L_D', 'angle_mie_deg', 'len_mie_L_D']

⚠ Warning: Some NaNs remain after interpolation:
Pc_bar              121
Tc_K                121
Pinj_bar            121
rho_kgm3            121
mu_Pas              121
angle_shadow_deg    121
len_shadow_L_D      121
angle_mie_deg       121
len_mie_L_D         121
dtype: int64


,run,Time_ms,Pc_bar,Tc_K,Pinj_bar,rho_kgm3,mu_Pas,angle_shadow_deg,len_shadow_L_D,angle_mie_deg,len_mie_L_D
0,ETH-01,0.000,55.03180,192.029519,98.864550,810.720228,0.001879,16.694545,13.126559,12.937325,17.571262
1,ETH-01,0.025,55.00570,192.015831,98.874062,810.718262,0.001879,16.694545,13.126559,12.937325,17.571262
2,ETH-01,0.050,55.00810,191.988228,98.907356,810.718443,0.001879,16.694545,13.126559,12.937325,17.571262
3,ETH-01,0.075,55.01635,192.081988,98.855037,810.719065,0.001879,16.694545,13.126559,12.937325,17.571262
4,ETH-01,0.100,55.01250,191.988000,98.878819,810.718775,0.001879,16.694545,20.204667,12.937325,24.506565


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 7 ▸ Final Validation & Export
# ─────────────────────────────────────────────────────────────

# Sort deterministically
filled = filled.sort_values(["run", "Time_ms"]).reset_index(drop=True)

# 1. Check for duplicates
duplicates = filled.duplicated(subset=["run", "Time_ms"])
if duplicates.any():
    print(f"⚠ Found {duplicates.sum()} duplicate rows! Dropping them...")
    filled = filled.drop_duplicates(subset=["run", "Time_ms"])

# 2. Check for expected columns
expected_cols = {
    "run", "Time_ms", "Pc_bar", "Tc_K", "Pinj_bar",
    "rho_kgm3", "mu_Pas", "angle_shadow_deg",
    "len_shadow_L_D", "angle_mie_deg", "len_mie_L_D"
}
missing_cols = expected_cols - set(filled.columns)
if missing_cols:
    print(f"⚠ Warning: Missing columns: {missing_cols}")

# 3. Save to CSV
filled.to_csv(OUTPUT_CSV, index=False)

print("─────────────────────────────────────────────────────────────")
print("✔ Successfully saved preprocessed data to:")
print(f"  {OUTPUT_CSV.resolve()}")
print("─────────────────────────────────────────────────────────────")
print(f"Final Shape : {filled.shape}")
print(f"Runs Found  : {filled['run'].nunique()}")
print(f"Run List    : {sorted(filled['run'].unique())}")
print("─────────────────────────────────────────────────────────────")

─────────────────────────────────────────────────────────────
✔ Successfully saved preprocessed data to:
  D:\GitHub Repos\spray-vision\data\processed\preprocessed_dataset.csv
─────────────────────────────────────────────────────────────
Final Shape : (847, 11)
Runs Found  : 7
Run List    : ['ETH-01', 'ETH-02', 'ETH-03', 'ETH-04', 'ETH-05', 'ETH-06', 'ETH-06.1']
─────────────────────────────────────────────────────────────
